# 05 · Data and integrations on GCP

How data and credentials flow through your deployment: files and dataframes on **GCS**,
**secrets** (Kubernetes-backed), the **BigQuery** connector, and consuming tasks published
by other teams.

**Learning goals**

1. Pass files, directories, and dataframes between tasks via GCS — without loading them into task inputs/outputs
2. Reference data that already lives in your buckets (`gs://...`)
3. Inject secrets as env vars or file mounts; understand where they live
4. Run parameterized BigQuery queries as connector tasks
5. Compose with other teams' deployed tasks via `flyte.remote.Task.get`

In [ ]:
import flyte
from flyte.io import Dir, File

flyte.init_from_config()

env = flyte.TaskEnvironment(
    name="gcp_data",
    resources=flyte.Resources(cpu="1", memory="1Gi"),
)

## 1. Files and directories

`File`/`Dir` are **references to objects in GCS**, not the bytes themselves. Returning a
`File` uploads once and passes a pointer; downstream tasks stream or download on demand.
This is the single most important I/O rule in Flyte:

> Large data goes through `File`/`Dir`/`DataFrame`. Task inputs/outputs (strings, lists,
> dicts) are for *coordination*, not payloads — oversized inline values slow every hop and
> can OOM the orchestrator.

Under the hood these land in the metadata bucket your platform team configured; task pods
reach GCS through Workload Identity (the `union` KSA → GSA binding) — no keys anywhere.

> 💬 **Discuss:** which GCS buckets and BigQuery datasets will your pipelines touch first, and which identities (KSA→GSA bindings) own that access today?

In [ ]:
@env.task
async def write_report(rows: int) -> File:
    with open("report.csv", "w") as f:
        f.write("id,value\n")
        for i in range(rows):
            f.write(f"{i},{i * 2}\n")
    return await File.from_local("report.csv")     # uploads to GCS, returns a reference


@env.task
async def summarize(report: File) -> int:
    total = 0
    async with report.open("r") as fh:              # streams from GCS
        content = await fh.read()
    for line in content.splitlines()[1:]:
        total += int(line.split(",")[1])
    return total


@env.task
async def report_pipeline(rows: int = 100) -> int:
    return await summarize(report=await write_report(rows=rows))


run = await flyte.run.aio(report_pipeline, rows=100)
print(run.url)
await run.wait.aio()
run.outputs()

In [ ]:
# Data that ALREADY lives in your buckets — reference it, nothing is copied:
existing = File.from_existing_remote("gs://YOUR-BUCKET/path/to/data.csv")

# Create a remote reference to write to, then hand it downstream:
# out = File.new_remote()

# Directories work the same way:
# d = Dir.from_existing_remote("gs://YOUR-BUCKET/training-data/")
# async for f in d.walk(): ...

## 2. DataFrames

`flyte.io.DataFrame` is a lazy, format-aware reference (Parquet on GCS by default) that
converts at the edges — so a pandas producer can feed a polars consumer:

In [ ]:
from flyte.io import DataFrame

df_image = (
    flyte.Image.from_debian_base(name="workshop-df", python_version=(3, 12))
    .with_pip_packages("pandas==2.2.3", "pyarrow")
)

df_env = flyte.TaskEnvironment(
    name="dataframes",
    image=df_image,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
)


@df_env.task
async def make_frame(rows: int) -> DataFrame:
    import pandas as pd
    pdf = pd.DataFrame({"id": range(rows), "value": [i * 2 for i in range(rows)]})
    return DataFrame.from_df(pdf)


@df_env.task
async def frame_stats(df: DataFrame) -> float:
    import pandas as pd
    pdf = await df.open(pd.DataFrame).all()
    return float(pdf["value"].mean())

## 3. Secrets

Secrets are stored by the platform (Kubernetes secrets in your dataplane; external stores
can back them — ask your platform team) and **injected at task startup** as env vars or
file mounts. Code never sees the secret store.

```bash
flyte create secret WORKSHOP_API_KEY my-secret-value
flyte create secret --from-file tls.crt CA_CERT
flyte get secret
```

In [ ]:
import os

secret_env = flyte.TaskEnvironment(
    name="with_secrets",
    resources=flyte.Resources(cpu="1", memory="512Mi"),
    secrets=[
        flyte.Secret(key="WORKSHOP_API_KEY", as_env_var="API_KEY"),
        # file mount, for certs/keys that shouldn't be in env vars:
        # flyte.Secret(key="CA_CERT", mount="/etc/workshop/secrets"),
    ],
)


@secret_env.task
async def call_protected_api() -> str:
    key = os.environ["API_KEY"]              # injected by the platform
    return f"authenticated with key of length {len(key)}"

Registry credentials for private images use the same machinery — a secret name passed as
`registry_secret` to `flyte.Image` ([01](./01-authoring-fundamentals.ipynb) §3). Build-time
secrets (private PyPI, private git) also ride on Flyte secrets — see the Union docs
*image building* pages for `with_pip_packages(..., secret_mounts=...)`.

## 4. BigQuery connector

Connector tasks **don't run in your container** — the query executes on the connector
service in the dataplane, which submits it to BigQuery's Jobs API and polls without
holding a worker. You declare the task; results come back as a `DataFrame`.

In [ ]:
from flyte.io import DataFrame
from flyteplugins.bigquery import BigQueryConfig, BigQueryTask
from workshop_config import WS                      # client-side values only

bq_config = BigQueryConfig(
    ProjectID=WS.bq_project or "YOUR-GCP-PROJECT",
    Location="US",
)

recent_events = BigQueryTask(
    name="recent_events",
    query_template="""
        SELECT name, state, year, SUM(number) AS total
        FROM `bigquery-public-data.usa_names.usa_1910_2013`
        WHERE state = @state AND year >= @min_year
        GROUP BY name, state, year
        ORDER BY total DESC
        LIMIT 20
    """,
    plugin_config=bq_config,
    inputs={"state": str, "min_year": int},          # typed query parameters
    output_dataframe_type=DataFrame,
)

bq_env = flyte.TaskEnvironment.from_task("bq_env", recent_events)

In [ ]:
run = await flyte.run.aio(recent_events, state="TX", min_year=2000)
print(run.url)
await run.wait.aio()

> **Auth on your deployment:** by default the connector uses the dataplane's Workload
> Identity. To use a dedicated service account instead, store its JSON key as a Flyte
> secret and pass `google_application_credentials="<secret-name>"` on the `BigQueryTask`.
>
> **Platform prerequisite:** the connector service must be enabled in the dataplane Helm
> values — [appendix A](./appendix/A-platform-config-map.md) → *Connectors*. Custom
> connectors (for internal services) are themselves deployable apps; see
> `user-guide/build-apps/connector-app` in the docs.

## 5. Cross-team composition: remote tasks

Teams publish tasks by deploying them; consumers reference by **name**, without importing
code or dependencies. First deploy the example shared task (once):

In [ ]:
!python scripts/remote_task_deploy.py

In [ ]:
import flyte.remote

tokenize = flyte.remote.Task.get("shared_utils.tokenize", auto_version="latest")


@env.task
async def use_shared(text: str) -> int:
    tokens = await tokenize(text=text)
    return len(tokens)


run = await flyte.run.aio(use_shared, text="composed from another team's task")
print(run.url)
await run.wait.aio()
run.outputs()

## Further reading

- Next: [07-apps-and-inference](./07-apps-and-inference.ipynb)